In [1]:
%pip install langchain langchain-community faiss-cpu
%pip install -U "docling-slim[format-pdf-docling]" \
    "docling-ibm-models[opencv-python-headless]" \
    langchain-docling

%pip install -qU langchain-ollama 

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------- ----------------------- 0.5/1.3 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 4.6 MB/s  0:00:00
  Attempting uninstall: docling-slim
    Found existing installation: docling-slim 2.150.1
    Uninstalling docling-slim-2.150.1:
      Successfully uninstalled docling-slim-2.150.1
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

C:\Users\ABHI MISHRA\AppData\Local\Temp\ipykernel_22876\2212358301.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
from langchain_docling import DoclingLoader
from docling.chunking import HybridChunker

FILE_PATH = "SIP (3).pdf"

loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type="doc_chunks",
    chunker=HybridChunker()
)

docs = loader.load()
if not docs:
    raise RuntimeError("Docling extracted no text from the PDF.")
    
print(f"Number of chunks: {len(docs)}")

D:\anaconda\envs\profiling\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
No OCR engine found. Please review the install details.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1777.14it/s]
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1203 > 512). Running this sequence through 

Number of chunks: 48


In [4]:
chunks = docs
for c in chunks:
    dl = c.metadata.get("dl_meta")
    if dl is None:          # already cleaned (cell re-run)
        continue
    c.metadata = {
        "source": c.metadata.get("source"),
        "pages": sorted({p["page_no"] for it in dl["doc_items"] for p in it["prov"]}),
        "section": " > ".join(dl.get("headings") or []),
    }
len(chunks)

48

In [5]:
print(chunks[0].metadata)
print(chunks[0].page_content[:300])

{'source': 'SIP (3).pdf', 'pages': [1], 'section': 'Transformer-Based Super-Resolution for Restoration of Degraded Historical Document Images'}
Transformer-Based Super-Resolution for Restoration of Degraded Historical Document Images
Sirigiri Venkateswara Adithya Abhi Mishra Ranveer Singh Mourya
Abstract-Historical handwritten documents often suffer from issues such as low resolution, faded ink, noise, and compression artifacts. These probl


In [6]:
from langchain_ollama import OllamaEmbeddings
embeddings=OllamaEmbeddings(
    model="qwen3-embedding:0.6b"
)

In [7]:
loaded_db=FAISS.from_documents(
    chunks,
    embeddings
)

In [8]:
results=loaded_db.similarity_search(
    "Give a brief summary about the paper",
    k=5
)

In [9]:
import ollama
from ollama import chat

response = chat(
    model='llama3.1:8b',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)
print(response.message.content)

Hello! How can I assist you today?


In [10]:
for d in results:
    print(d.metadata["pages"], "|", d.metadata["section"])
    print(d.page_content[:200].replace("\n", " "), "\n")

[6] | REPRODUCIBILITY
REPRODUCIBILITY Code (Google Colab): Colab Notebook. Kaggle notebooks, checkpoints and result files: Google Drive Folder. 

[6] | VIII. CONCLUSION
VIII. CONCLUSION | - |  R0. Ours (E2), Setting = L1+Sobel. Ours (E2), Scale = 4×. Ours (E2), PSNR = 39.2474. Ours (E2), SSIM = 0.9621. Ours (E2), Direct? = -. Ours (E3), Year = 2026. Ours (E3), Datase 

[5, 6] | VIII. CONCLUSION
VIII. CONCLUSION We presented a controlled study of transformer-based 4× super-resolution for historical handwritten documents on Bentham R0. A transformer trained with L1 loss already outperforms bil 

[6] | VIII. CONCLUSION
VIII. CONCLUSION | - |  N/R. AI-driven enhancement of historical documents, SSIM = N/R. AI-driven enhancement of historical documents, Direct? = No. Lanczos (ours, baseline), Year = 2026. Lanczos (our 

[2] | A. Bentham R0
A. Bentham R0 We use the Bentham R0 collection of historical handwritten pages [10]. It provides 433 page images with 433 matching PAGE XML files [11] th

In [11]:
prompt=PromptTemplate(
    template="""
      You are a helpful research assistant.
      Answer only from the provided context.
      If the context is insufficient then just say don't know about it.
      When evidence is available, mention the source page for example(p.3).

      Context:{context}
      Question: {question}
    """,
    input_variables=['context','question']
)

In [25]:

def retrieve_context(question: str, k: int = 7)->str:
    docs = loaded_db.similarity_search(question, k=k)
    context = "\n\n---\n\n".join(
         f"[Section: {d.metadata['section']} | Pages: {', '.join(map(str, d.metadata['pages']))}]\n{d.page_content}"
        for d in docs
    )
    return context
    


In [27]:
from langchain_ollama import ChatOllama

In [28]:
  llm = ChatOllama(
        model="llama3.1:8b",
        temperature= 0.1)

In [29]:
parser=StrOutputParser()

In [30]:
parallel_chain=RunnableParallel(
    {
        'context':RunnableLambda(retrieve_context),
        'question':RunnablePassthrough()
    }
)

In [31]:
main_chain=parallel_chain|prompt|llm|parser

In [32]:
result=main_chain.invoke("Which technique is ued for super resolution of historical images? explain in detail")
result

'According to the provided context, the technique used for super-resolution of historical images is a patch-based transformer encoder-decoder architecture inspired by DocEnTr.\n\nIn the introduction section (p.1), it is mentioned that the authors investigated three questions related to super-resolution of historical document images. The first question (RQ1) is: "Can a transformer-based model reconstruct historical document images better than conventional interpolation?"\n\nIn the section "A. Single-Image Super-Resolution" (p.1), it is mentioned that deep SR began with convolutional networks such as SRCNN [1] and deeper residual designs such as EDSR [2]. However, the authors of this study used a transformer-based approach, which is a different technique.\n\nIn the section "The first experiment (E1) used a baseline model trained with L1 loss. In the second experiment (E2), we introduced a Sobel edge-preservation loss to help the model retain important document details and edges. The thir